# Goal-pruning trajectories: what the model sees at each step

`GoalPruningCodeAgent` keeps two lists and the prompt is their concatenation:

```
prompt history = committed_steps + current_context
                 ^ one survivor    ^ every step of the
                   per closed        subtask still in
                   subtask           progress
```

Per step the model states an **Expectation** before writing code, a judge scores whether the
observation delivered it, and **only a judged success prunes**:

| outcome | what happens |
|---|---|
| `p >= threshold` | subtask closes: this step is committed, every open step before it is **dropped from the prompt forever** |
| `p < threshold` | step joins the open set; nothing is dropped |
| no Expectation stated, or final-answer step | not judged, `p` is `None`, never commits |
| step raised an error | `p = 0.0` without spending a judge call |

There is no step-count cap, so a trajectory whose expectations are never satisfied keeps its whole
history and runs as plain ReAct. That is the floor this design degrades to.

A second, less obvious floor: **a commit only discards something if its subtask ran more than one
step.** If a step satisfies its own expectation immediately, it closes a one-step subtask and drops
nothing -- so a 100% commit rate prunes exactly as little as a 0% one. On gpt-4o, 45 of 165
questions never committed, but 79 ended up dropping nothing.

Because `completion_prob` is recorded per step, the buffer can be replayed exactly offline -- no
model calls. That is what the renderer below does, so every step shows the step numbers that were
actually in its prompt, and each commit shows which steps it discarded.

In [6]:
# ===================== what to look at =====================
MODEL = "gpt-4o"        # key of GP_RUNS below
PKL_INDEX = 0         # GAIA validation index; the last cell ranks the interesting ones
THRESHOLD = 0.5         # must match the run's completion_threshold to replay it faithfully
# ===========================================================

import pickle
from pathlib import Path

GP_RUNS = {
    "gpt-4o": Path("goalprune_gp1_react_gpt-4o"),
    "gpt-5.4-mini": Path("goalprune_gp1_react_gpt-5.4-mini"),
    "Qwen/Qwen3.5-9B": Path("goalprune_gp1_react_Qwen/Qwen3.5-9B"),
    "Qwen/Qwen3.7-Plus": Path("goalprune_gp1_react_Qwen/Qwen3.7-Plus"),
}
for name, d in GP_RUNS.items():
    print(f"{len(list(d.glob('*.pkl'))) if d.exists() else 0:>4}/165  {name}")

RUN_DIR = GP_RUNS[MODEL]
row = pickle.load(open(RUN_DIR / f"{PKL_INDEX}.pkl", "rb"))
print(f"\nq{PKL_INDEX}: {row['num_steps']} steps | correct={row['is_correct']} | "
      f"{row['token_counts']['total_tokens']:,} tokens | error={row.get('error')}")

 165/165  gpt-4o
 165/165  gpt-5.4-mini
 165/165  Qwen/Qwen3.5-9B
 165/165  Qwen/Qwen3.7-Plus

q0: 51 steps | correct=False | 646,723 tokens | error=None


In [7]:
# Replay the buffer offline. Only a judged success prunes, so this reproduces the agent exactly.
import html

from IPython.display import HTML, display

OBS_LIMIT = 3000
CODE_LIMIT = 700
THOUGHT_LIMIT = 600


def replay(steps, threshold=THRESHOLD):
    """Per step: what was visible in its prompt, and what its outcome dropped."""
    committed, open_, rows = [], [], []
    for s in steps:
        committed_before = [x["step_number"] for x in committed]
        open_before = [x["step_number"] for x in open_]
        open_.append(s)
        p = s.get("completion_prob")
        satisfied = not s.get("is_final_answer") and p is not None and p >= threshold
        dropped = []
        if satisfied:
            dropped = [x["step_number"] for x in open_[:-1]]
            committed.append(s)
            open_ = []
        rows.append(
            {
                "step": s,
                "committed_before": committed_before,
                "open_before": open_before,
                "satisfied": satisfied,
                "dropped": dropped,
                "committed_after": [x["step_number"] for x in committed],
                "open_after": [x["step_number"] for x in open_],
            }
        )
    return rows


def _verdict(s, satisfied):
    """Why this step got the probability it did -- the judge is skipped in three cases."""
    p = s.get("completion_prob")
    if s.get("is_final_answer"):
        return "final answer &mdash; not judged", "#546e7a"
    if s.get("error"):
        return "step errored &mdash; unsatisfied, no judge call", "#c62828"
    if p is None:
        return "no Expectation stated &mdash; not judged, can never commit", "#ef6c00"
    if satisfied:
        return f"SATISFIED (p={p:.4f}) &mdash; subtask closed", "#2e7d32"
    return f"not satisfied (p={p:.4g}) &mdash; stays open", "#5c6bc0"


def _bar(p, colour="#26a69a"):
    if p is None:
        return "<span style='color:#90a4ae'>n/a</span>"
    width = max(1, round(p * 120))
    return (
        f"<span style='display:inline-block;width:120px;background:#eceff1;vertical-align:middle'>"
        f"<span style='display:inline-block;width:{width}px;height:9px;background:{colour}'></span></span>"
        f" <code>{p:.6g}</code>"
    )


def _block(label, text, limit, colour):
    if not text:
        return ""
    body = html.escape(str(text)[:limit])
    more = "" if len(str(text)) <= limit else f"<div style='color:#90a4ae'>... {len(str(text)) - limit:,} more chars</div>"
    return (
        f"<div style='margin:6px 0'><div style='font:600 11px sans-serif;color:{colour}'>{label}</div>"
        f"<pre style='margin:2px 0;padding:6px 8px;background:#fafafa;border-left:3px solid {colour};"
        f"white-space:pre-wrap;font-size:11.5px'>{body}{more}</pre></div>"
    )


def _pills(numbers, colour):
    if not numbers:
        return "<span style='color:#b0bec5'>none</span>"
    return " ".join(
        f"<span style='background:{colour};color:#fff;border-radius:3px;padding:1px 6px;"
        f"font:600 11px monospace'>{n}</span>"
        for n in numbers
    )

In [8]:
def render_goal_trajectory(row, threshold=THRESHOLD):
    """One panel per step: its prompt contents, its expectation, and what its verdict pruned."""
    rows = replay(row["steps"], threshold)
    parts = [
        f"<div style='font:13px sans-serif;max-width:1100px'>"
        f"<div style='padding:8px;background:#eceff1'><b>q{PKL_INDEX}</b> &mdash; {html.escape(row['question'][:300])}"
        f"<br><b>true</b>: {html.escape(str(row['true_answer']))}"
        f"<br><b>predicted</b>: {html.escape(str(row['prediction'])[:300])}"
        f" &nbsp; <b>correct</b>: {row['is_correct']}"
        f"<br><b>threshold</b>: {threshold} &nbsp; <b>steps</b>: {len(rows)}</div>"
    ]
    for r in rows:
        s = r["step"]
        label, colour = _verdict(s, r["satisfied"])
        n = s["step_number"]
        parts.append(
            f"<div style='border:1px solid #cfd8dc;border-left:5px solid {colour};margin:14px 0;padding:8px'>"
            f"<div style='font:600 14px sans-serif'>Step {n}"
            f"<span style='float:right;color:{colour}'>{label}</span></div>"
            f"<div style='background:#f5f5f5;padding:6px;margin:6px 0;font-size:12px'>"
            f"<b>in the prompt for this step</b><br>"
            f"committed (survivors of closed subtasks): {_pills(r['committed_before'], '#2e7d32')}<br>"
            f"open (this subtask so far): {_pills(r['open_before'], '#78909c')}"
            f"</div>"
            f"<div style='margin:4px 0'><b>judge</b> {_bar(s.get('completion_prob'))}</div>"
            + _block("Expectation", s.get("expectation") or "(none stated)", 400, "#ef6c00")
            + _block("Thought", s.get("model_output"), THOUGHT_LIMIT, "#7e57c2")
            + _block("Code", s.get("code_action"), CODE_LIMIT, "#5c6bc0")
            + _block("Observation", s.get("observations") or s.get("error"), OBS_LIMIT, "#90a4ae")
            + (
                f"<div style='background:#e8f5e9;padding:6px;margin-top:6px;font-size:12px'>"
                f"<b>commit</b> &mdash; step {n} kept. dropped from the prompt: {_pills(r['dropped'], '#c62828')}"
                f"<br>committed now: {_pills(r['committed_after'], '#2e7d32')} &nbsp;"
                f"open now: {_pills(r['open_after'], '#78909c')}</div>"
                if r["satisfied"]
                else f"<div style='padding:6px;margin-top:6px;font-size:12px;color:#546e7a'>"
                f"open set is now {_pills(r['open_after'], '#78909c')} &mdash; nothing dropped</div>"
            )
            + "</div>"
        )
    final = rows[-1] if rows else None
    if final:
        visible = len(final["committed_after"]) + len(final["open_after"])
        parts.append(
            f"<div style='padding:8px;background:#eceff1'><b>end of run</b>: {visible} of {len(rows)} steps "
            f"still in the prompt ({visible / len(rows):.0%} retained). "
            f"committed {_pills(final['committed_after'], '#2e7d32')} open {_pills(final['open_after'], '#78909c')}</div>"
        )
    display(HTML("".join(parts) + "</div>"))
    return rows


rows = render_goal_trajectory(row)

In [9]:
# Compact timeline: one line per step, so the pruning pattern is visible at a glance.
import pandas as pd

table = pd.DataFrame(
    [
        {
            "step": r["step"]["step_number"],
            "p(satisfied)": r["step"].get("completion_prob"),
            "committed?": "COMMIT" if r["satisfied"] else "",
            "visible in prompt": len(r["committed_before"]) + len(r["open_before"]),
            "dropped here": ",".join(str(x) for x in r["dropped"]),
            "expectation": (r["step"].get("expectation") or "(none)")[:60],
        }
        for r in rows
    ]
)
commits = sum(1 for r in rows if r["satisfied"])
dropped = sum(len(r["dropped"]) for r in rows)
print(f"q{PKL_INDEX}: {len(rows)} steps | {commits} commits | {dropped} steps dropped | "
      f"{len(rows) - dropped} still in the prompt at the end")
table

q0: 51 steps | 2 commits | 21 steps dropped | 30 still in the prompt at the end


,step,p(satisfied),committed?,visible in prompt,dropped here,expectation
0,1,0.000000e+00,,0,,A list of the words labeling the axes of the f...
1,2,1.015659e-06,,1,,Find a link or reference to an AI regulation p...
2,3,0.000000e+00,,2,,A link to the AI regulation paper submitted to...
3,4,3.957549e-07,,3,,Link to or details of a paper about AI regulat...
4,5,0.000000e+00,,4,,Find and access a paper about AI regulation fr...
5,6,0.000000e+00,,5,,Find and access the Physics and Society articl...
6,7,1.972334e-07,,6,,Locate a link or description of the 2016 Physi...
7,8,2.570913e-06,,7,,Locate high-priority articles relevant to AI a...
8,9,0.000000e+00,,8,,(none)
9,10,7.211037e-01,COMMIT,9,"1,2,3,4,5,6,7,8,9",Provide guidance based on the current situatio...


In [10]:
# Which questions are worth opening? Rank by how much was actually pruned.
summary = []
for f in sorted(RUN_DIR.glob("*.pkl"), key=lambda p: int(p.stem)):
    r = pickle.load(open(f, "rb"))
    if not r["steps"]:
        continue
    rp = replay(r["steps"])
    dropped = sum(len(x["dropped"]) for x in rp)
    summary.append(
        {
            "idx": int(f.stem),
            "steps": len(rp),
            "commits": sum(1 for x in rp if x["satisfied"]),
            "dropped": dropped,
            "retained": round(1 - dropped / len(rp), 2),
            "correct": bool(r["is_correct"]),
            "tokens": r["token_counts"]["total_tokens"],
        }
    )

summary = pd.DataFrame(summary).sort_values(["dropped", "steps"], ascending=False)
print(f"{RUN_DIR.name}: {len(summary)} questions")
print(f"  no commit at all (judge never satisfied)  : {(summary['commits'] == 0).sum()}")
print(f"  nothing dropped from the prompt           : {(summary['dropped'] == 0).sum()}  <- the plain-ReAct floor")
print(f"  mean retention                            : {summary['retained'].mean():.0%}")
print("\nThose two counts differ because a commit only discards something if its subtask was")
print("more than one step long. A step that satisfies its own expectation immediately closes a")
print("one-step subtask and drops nothing -- full commit rate, zero pruning.")
print("\nMost pruning happened on these -- set PKL_INDEX above:")
summary.head(15)

goalprune_gp1_react_gpt-4o: 165 questions
  no commit at all (judge never satisfied)  : 46
  nothing dropped from the prompt           : 80  <- the plain-ReAct floor
  mean retention                            : 75%

Those two counts differ because a commit only discards something if its subtask was
more than one step long. A step that satisfies its own expectation immediately closes a
one-step subtask and drops nothing -- full commit rate, zero pruning.

Most pruning happened on these -- set PKL_INDEX above:


,idx,steps,commits,dropped,retained,correct,tokens
24,24,51,2,48,0.06,False,742686
155,155,51,8,36,0.29,False,557431
29,29,30,1,27,0.10,False,460745
132,132,29,3,25,0.14,True,492497
159,159,27,1,22,0.19,False,322408
0,0,51,2,21,0.59,False,646723
100,100,51,4,20,0.61,False,778395
121,121,22,2,18,0.18,False,163369
12,12,20,2,16,0.20,False,245766
31,31,23,3,14,0.39,False,254816
